## Notebook to start a spark instance in an ArcGIS Pro Session.

### THE BELOW HAS TO BE THE FIRST IMPORT !

In [1]:
from spark_esri import spark_start, spark_stop

In [2]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

In [3]:
import os

import arcpy
import geofunctions as S

### Check if the jar exists

In [5]:
spark_jars = os.path.expanduser("~/geofunctions-0.10.jar")
os.path.exists(spark_jars)

True

In [6]:
config = {
    "spark.driver.memory": "32G",
    "spark.executor.memory": "32G",
    "spark.jars": spark_jars,
}
spark = spark_start(config=config)

In [7]:
spark.version

'3.5.1-esri-1'

### Sample usage. Create random points in the current map extent and place them on the Map.

In [8]:
project = arcpy.mp.ArcGISProject("CURRENT")

extent = project.activeView.camera.getExtent()
xmin = extent.XMin
ymin = extent.YMin
dx = extent.XMax - extent.XMin
dy = extent.YMax - extent.YMin

In [9]:
(
    spark
    # Create 1000 points.
    .range(1000)
    # Get random x values
    .withColumn("x", F.rand() * dx + xmin)
    # Get random y values.
    .withColumn("y", F.rand() * dy + ymin)
    # Create a point geometry from implicit x/y.
    .withColumn("geom", S.st_point())
    # Convert the spark dataframe to a feature class to be displayed in the TOC.
    .to_feature_class(
        "MyPoints",
        sp_ref = -1,  # Use current map SR.
    )
)